In [2]:
import os
import sys
import cv2
import json
import numpy as np
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
import cv2
import os
import math


In [3]:

BASE_DIR = os.getcwd()
VAL_DIR = os.path.join(BASE_DIR, "dataset/images/val")

TOTAL_IMAGES = 40
BRIGHTEST_COUNT = 5
DARKEST_COUNT = 5
MOST_ROAD_COUNT = 5
MOST_LINES_COUNT = 5
MOST_OBJECTS_COUNT = 10


In [4]:

def compute_luminance(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return 0.0
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return np.mean(gray)


In [5]:

def compute_road_pixels(img_path):
    da_path = img_path.replace("images", "da_seg_annotations").replace(".jpg", ".png")
    if not os.path.exists(da_path):
        return 0
    mask = cv2.imread(da_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return 0
    return np.sum(mask > 0)


In [6]:

def compute_line_pixels(img_path):
    ll_path = img_path.replace("images", "ll_seg_annotations").replace(".jpg", ".png")
    if not os.path.exists(ll_path):
        return 0
    mask = cv2.imread(ll_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return 0
    return np.sum(mask > 0)


In [7]:

def compute_object_count(img_path):
    det_path = img_path.replace("images", "det_annotations").replace(".jpg", ".json")
    if not os.path.exists(det_path):
        return 0
    try:
        with open(det_path, 'r') as f:
            data = json.load(f)
        frames = data.get('frames', [])
        if frames:
            objects = frames[0].get('objects', [])
            return len([o for o in objects if o.get('category') == 'car'])
    except:
        return 0
    return 0


In [8]:

def generate_calibration_dataset(
    val_dir,
    total_images=50,
    brightest_count=6,
    darkest_count=6,
    most_road_count=6,
    most_lines_count=6,
    most_objects_count=12
):
    image_files = sorted([
        os.path.join(val_dir, f)
        for f in os.listdir(val_dir)
        if f.endswith('.jpg')
    ])
    
    if len(image_files) == 0:
        print(f"No images found in {val_dir}")
        return []
    
    print(f"Analyzing {len(image_files)} images...")
    
    luminance_data = []
    road_data = []
    line_data = []
    object_data = []
    
    for img_path in tqdm(image_files, desc="Computing metrics"):
        luminance = compute_luminance(img_path)
        road_pixels = compute_road_pixels(img_path)
        line_pixels = compute_line_pixels(img_path)
        object_count = compute_object_count(img_path)
        
        luminance_data.append((img_path, luminance))
        road_data.append((img_path, road_pixels))
        line_data.append((img_path, line_pixels))
        object_data.append((img_path, object_count))
    
    selected_images = set()
    
    luminance_sorted = sorted(luminance_data, key=lambda x: x[1], reverse=True)
    for i in range(min(brightest_count, len(luminance_sorted))):
        selected_images.add(luminance_sorted[i][0])
    
    luminance_sorted_asc = sorted(luminance_data, key=lambda x: x[1])
    for i in range(min(darkest_count, len(luminance_sorted_asc))):
        selected_images.add(luminance_sorted_asc[i][0])
    
    road_sorted = sorted(road_data, key=lambda x: x[1], reverse=True)
    for i in range(min(most_road_count, len(road_sorted))):
        selected_images.add(road_sorted[i][0])
    
    line_sorted = sorted(line_data, key=lambda x: x[1], reverse=True)
    for i in range(min(most_lines_count, len(line_sorted))):
        selected_images.add(line_sorted[i][0])
    
    object_sorted = sorted(object_data, key=lambda x: x[1], reverse=True)
    for i in range(min(most_objects_count, len(object_sorted))):
        selected_images.add(object_sorted[i][0])
    
    remaining_count = total_images - len(selected_images)
    if remaining_count > 0:
        available_images = [img for img in image_files if img not in selected_images]
        random_selection = random.sample(available_images, min(remaining_count, len(available_images)))
        selected_images.update(random_selection)
    
    final_list = sorted(list(selected_images))[:total_images]
    
    basename_list = [os.path.basename(img) for img in final_list]
    
    return basename_list


In [9]:
def display_calibration_images(image_list, val_dir, grid_cols=5):

    
    num_images = len(image_list)
    grid_rows = math.ceil(num_images / grid_cols)
    
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(grid_cols * 4, grid_rows * 3))
    
    if grid_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, img_name in enumerate(image_list):
        row = idx // grid_cols
        col = idx % grid_cols
        
        img_path = os.path.join(val_dir, img_name)
        
        if os.path.exists(img_path):
            img_bgr = cv2.imread(img_path)
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            
            axes[row, col].imshow(img_rgb)
            axes[row, col].set_title(img_name, fontsize=8)
            axes[row, col].axis('off')
        else:
            axes[row, col].text(0.5, 0.5, 'Not Found', ha='center', va='center')
            axes[row, col].set_title(img_name, fontsize=8)
            axes[row, col].axis('off')
    
    for idx in range(num_images, grid_rows * grid_cols):
        row = idx // grid_cols
        col = idx % grid_cols
        axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:

if __name__ == "__main__":
    calibration_dataset = generate_calibration_dataset(
        VAL_DIR,
        total_images=TOTAL_IMAGES,
        brightest_count=BRIGHTEST_COUNT,
        darkest_count=DARKEST_COUNT,
        most_road_count=MOST_ROAD_COUNT,
        most_lines_count=MOST_LINES_COUNT,
        most_objects_count=MOST_OBJECTS_COUNT
    )
    
    print(f"\nCalibration dataset generated: {len(calibration_dataset)} images")
    print("\nCopy-paste ready format:")
    print(calibration_dataset)
    
    display_calibration_images(calibration_dataset, VAL_DIR, grid_cols=5)

Analyzing 10000 images...


Computing metrics: 100%|██████████████████████████████████████████████████████████████| 10000/10000 [03:54<00:00, 42.69it/s]



Calibration dataset generated: 40 images

Copy-paste ready format:
['b2932832-df9384eb.jpg', 'b2afce7f-0cb9967a.jpg', 'b3770858-ea33d0dc.jpg', 'b4946e87-f31e29b1.jpg', 'b49a272b-c52895c3.jpg', 'b5183558-782ad31c.jpg', 'b5770084-9b15155d.jpg', 'b5a50c2d-ccafe43a.jpg', 'b64e5796-3aa7adc0.jpg', 'b780088d-b85f1b00.jpg', 'b792daf1-e95167d2.jpg', 'b7d13f97-3df013fa.jpg', 'b7f43f41-b1354d92.jpg', 'b90d0148-bbecdb11.jpg', 'b9d12c48-d855b440.jpg', 'ba65317f-d4aba330.jpg', 'bb3bb8c0-2ef3abc6.jpg', 'bb6e9fc4-35369ff6.jpg', 'bcd8fde8-51118aa7.jpg', 'bcf7f6b9-88bfed8f.jpg', 'bd5f8f37-b3008fa1.jpg', 'be776742-fe6599b2.jpg', 'bea82122-66fc59a4.jpg', 'bf1a30f8-59a115d8.jpg', 'c019c1f1-883ea964.jpg', 'c0c9ec9a-dad1653f.jpg', 'c1ba5ee6-1a00a269.jpg', 'c2341eec-16d212b8.jpg', 'c28cecbb-11796cf6.jpg', 'c3698cec-77311999.jpg', 'c3a18207-504667c7.jpg', 'c4f4718b-5356b324.jpg', 'c54ec43c-1aa7875b.jpg', 'c57cda5b-7ca48687.jpg', 'c5b2506d-9121123c.jpg', 'c6d51f76-d2459db8.jpg', 'c7346130-9d225646.jpg', 'c75e2

### Liste generées :
['b371451e-7caf934b.jpg', 'b3e0345b-69069f01.jpg', 'b4715825-7c265849.jpg', 'b4946e87-f31e29b1.jpg', 'b49a272b-c52895c3.jpg', 'b4f4bd95-a097514a.jpg', 'b4f85748-c4be0117.jpg', 'b5183558-782ad31c.jpg', 'b5770084-9b15155d.jpg', 'b5a50c2d-ccafe43a.jpg', 'b64e5796-3aa7adc0.jpg', 'b737b268-3e81c0e9.jpg', 'b74e7806-60077058.jpg', 'b792daf1-e95167d2.jpg', 'b7f43f41-b1354d92.jpg', 'b81d060a-fddedd4f.jpg', 'b857eb06-a85b37f5.jpg', 'b99c5931-81b850be.jpg', 'ba65317f-d4aba330.jpg', 'bb3bb8c0-2ef3abc6.jpg', 'bb6e9fc4-35369ff6.jpg', 'bb9f498e-62cc407e.jpg', 'bc97328c-8b9ba7ab.jpg', 'bcd8fde8-51118aa7.jpg', 'bcf7f6b9-88bfed8f.jpg', 'bd39c71a-5f45bcc3.jpg', 'bd5f8f37-b3008fa1.jpg', 'be5ca360-3585455f.jpg', 'bea82122-66fc59a4.jpg', 'bf1a30f8-59a115d8.jpg', 'bf527384-71f7e998.jpg', 'c019c1f1-883ea964.jpg', 'c0c9ec9a-dad1653f.jpg', 'c1ba5ee6-1a00a269.jpg', 'c2341eec-16d212b8.jpg', 'c28cecbb-11796cf6.jpg', 'c3698cec-77311999.jpg', 'c3a18207-504667c7.jpg', 'c3b31662-4e576035.jpg', 'c4e97226-7be7eef5.jpg', 'c4f4718b-5356b324.jpg', 'c57cda5b-7ca48687.jpg', 'c6d51f76-d2459db8.jpg', 'c7346130-9d225646.jpg', 'c75e23fe-fbadb3fa.jpg', 'c7cece6c-3eccc86a.jpg', 'c891763b-fea20b04.jpg', 'c911f50f-e7958776.jpg', 'c92005f1-24f38b48.jpg', 'c962fce0-dcea4e6a.jpg']


['b371451e-7caf934b.jpg', 'b446d26a-9717e68c.jpg', 'b4946e87-f31e29b1.jpg', 'b49a272b-c52895c3.jpg', 'b5183558-782ad31c.jpg', 'b5770084-9b15155d.jpg', 'b5a50c2d-ccafe43a.jpg', 'b5fc1373-94aaab40.jpg', 'b64e5796-3aa7adc0.jpg', 'b66462a4-ce0558e1.jpg', 'b6a8503c-a797867c.jpg', 'b729d9e2-a4d95e09.jpg', 'b74e7806-60077058.jpg', 'b764f1b2-7ce1c79f.jpg', 'b792daf1-e95167d2.jpg', 'b7f43f41-b1354d92.jpg', 'b9ecc316-2fdb6c92.jpg', 'ba65317f-d4aba330.jpg', 'bb3bb8c0-2ef3abc6.jpg', 'bb6e9fc4-35369ff6.jpg', 'bb9f498e-62cc407e.jpg', 'bc709acb-5b7fbd2f.jpg', 'bcd8fde8-51118aa7.jpg', 'bcf7f6b9-88bfed8f.jpg', 'bd39c71a-5f45bcc3.jpg', 'bd5f8f37-b3008fa1.jpg', 'bea82122-66fc59a4.jpg', 'bf1a30f8-59a115d8.jpg', 'c019c1f1-883ea964.jpg', 'c0c9ec9a-dad1653f.jpg', 'c1ba5ee6-1a00a269.jpg', 'c2341eec-16d212b8.jpg', 'c28cecbb-11796cf6.jpg', 'c31a04de-d4cc31fb.jpg', 'c3698cec-2231cb86.jpg', 'c3698cec-77311999.jpg', 'c3a18207-504667c7.jpg', 'c3b31662-4e576035.jpg', 'c4f4718b-5356b324.jpg', 'c5430bf3-07e67721.jpg', 'c57cda5b-7ca48687.jpg', 'c6d51f76-d2459db8.jpg', 'c6dad47e-47a61b83.jpg', 'c7346130-9d225646.jpg', 'c75e23fe-fbadb3fa.jpg', 'c891763b-fea20b04.jpg', 'c8cbb112-9d16c5da.jpg', 'c92005f1-24f38b48.jpg', 'c928c559-b70b74cf.jpg', 'c962fce0-dcea4e6a.jpg']



['b2932832-df9384eb.jpg', 'b2afce7f-0cb9967a.jpg', 'b3770858-ea33d0dc.jpg', 'b4946e87-f31e29b1.jpg', 'b49a272b-c52895c3.jpg', 'b5183558-782ad31c.jpg', 'b5770084-9b15155d.jpg', 'b5a50c2d-ccafe43a.jpg', 'b64e5796-3aa7adc0.jpg', 'b780088d-b85f1b00.jpg', 'b792daf1-e95167d2.jpg', 'b7d13f97-3df013fa.jpg', 'b7f43f41-b1354d92.jpg', 'b90d0148-bbecdb11.jpg', 'b9d12c48-d855b440.jpg', 'ba65317f-d4aba330.jpg', 'bb3bb8c0-2ef3abc6.jpg', 'bb6e9fc4-35369ff6.jpg', 'bcd8fde8-51118aa7.jpg', 'bcf7f6b9-88bfed8f.jpg', 'bd5f8f37-b3008fa1.jpg', 'be776742-fe6599b2.jpg', 'bea82122-66fc59a4.jpg', 'bf1a30f8-59a115d8.jpg', 'c019c1f1-883ea964.jpg', 'c0c9ec9a-dad1653f.jpg', 'c1ba5ee6-1a00a269.jpg', 'c2341eec-16d212b8.jpg', 'c28cecbb-11796cf6.jpg', 'c3698cec-77311999.jpg', 'c3a18207-504667c7.jpg', 'c4f4718b-5356b324.jpg', 'c54ec43c-1aa7875b.jpg', 'c57cda5b-7ca48687.jpg', 'c5b2506d-9121123c.jpg', 'c6d51f76-d2459db8.jpg', 'c7346130-9d225646.jpg', 'c75e23fe-fbadb3fa.jpg', 'c891763b-fea20b04.jpg', 'c962fce0-dcea4e6a.jpg']